# Chapitre 10 - Cours 1 : traitement par lots avec les outils Unix

Les chapitres précédents parlaient de requêtes et de réponses : un client demande, le
système répond le plus vite possible. Le traitement **par lots** (*batch processing*)
change de point de vue : un job lit un gros volume de données d'entrée, calcule pendant
des minutes ou des heures, et écrit une sortie entièrement **dérivée** de son entrée. Ce
premier cours en pose l'intuition sur une seule machine, avec le plus ancien des
frameworks de traitement par lots : le shell Unix. Il couvre l'introduction du chapitre et
la section *Batch Processing with Unix Tools* (*Simple Log Analysis*, *Chain of commands versus custom program*, *Sorting versus in-memory aggregation*, *The Unix Philosophy*).

**Plan.**

1. Trois façons de traiter des données : en ligne, par lots, en flux.
2. Une analyse de journal avec `awk`, `sort`, `uniq` et `head`.
3. Chaîne de commandes contre programme : même résultat, exécution différente.
4. Trier ou agréger en mémoire : l'ensemble de travail et le tri externe.
5. Les tuyaux, un tampon borné et la contre-pression.
6. La philosophie Unix : interface uniforme, logique séparée du câblage, expérimentation.

**Contexte : la troisième partie du livre.** Ce chapitre ouvre la partie III, *Derived
Data*. Son introduction distingue deux familles de systèmes de données. Un **système de
référence** (*system of record*, la « source de vérité ») détient la version faisant foi
de chaque donnée : quand une information y entre, elle y est écrite une fois, et en cas
de désaccord c'est lui qui a raison. Un système de **données dérivées** contient le
résultat d'une transformation de données venues d'ailleurs : cache, index, vue
matérialisée, modèle de recommandation. Une donnée dérivée est redondante (on pourrait
la recalculer depuis la source) mais elle rend les lectures rapides. Le traitement par
lots est l'outil type pour construire de telles données dérivées.

In [1]:
import os
import shutil
import subprocess
import tempfile
import time
import tracemalloc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import batch as B
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 90)
WORK = tempfile.mkdtemp(prefix="ddia_ch10_")      # dossier de travail, effacé en fin de notebook
ENV_C = dict(os.environ, LC_ALL="C")                  # tri octet par octet : rapide et déterministe


def sh(cmd):
    """Lance une commande shell et renvoie sa sortie standard."""
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True, env=ENV_C).stdout